# Aufgabe 10.2 – AI4I mit Jupyter erkunden

**Status: AWAITING HUMAN REVIEW.** Initiale Implementierung und technische Prüfung:
Codex. Eine studentische Freigabe dieser Aufgabe liegt noch nicht vor.

Ziel: einen Datensatz laden, eine begrenzte deskriptive Übersicht erstellen und
Notebook-Zelltypen sowie HTML/PDF-Exporte verwenden.

Quelle der Aufgabe: [Industrial Data Science, Exercise 10.2](https://noxthot.github.io/ws26_industrial_datascience_manuscript/1_act1/6_self_study_session1.html).
Abruf: 26.10.2026. Ausführung aus dem Projektstamm oder aus diesem Notebook-Ordner.
Vorher: `uv sync --locked` und `uv run python scripts/download_ai4i.py`.
Danach ist die Notebookausführung ohne Netzwerkzugriff möglich.

## Datenherkunft und Umfang

[AI4I 2020 Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset),
UCI Machine Learning Repository (2020), [DOI 10.24432/C5HS5C](https://doi.org/10.24432/C5HS5C),
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Der Datensatz ist **synthetisch** und enthält laut Anbieter 10.000 Zeilen.
Originaldaten, Prüfsumme und Downloadweg sind in `data/raw/README.md` dokumentiert.

Wir behalten ausschließlich `UDI`, `Torque [Nm]`, `Rotational speed [rpm]`
und `Air temperature [K]`. UDI dient als Zeilenkennung, nicht als Zeitstempel.
Wir schätzen keine Modelle und treffen keine kausalen Aussagen.

In [ ]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ.setdefault("MPLCONFIGDIR", str(root / ".tools/matplotlib"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
from sessions.ch10.ai4i_data import COLUMNS, DATA_PATH, verify_file, validate_data

output_dir = root / "outputs/ch10/exercise_10_2"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Python {sys.version.split()[0]}; pandas {pd.__version__}; Matplotlib {matplotlib.__version__}")

## Laden und Datenqualität

Der Download erfolgt ausdrücklich vor der Analyse. Zuerst prüfen wir die exakten
Rohbytes mittels SHA256; dann liest pandas nur die vier geforderten Spalten.
Ihre Reihenfolge und Einheiten bleiben explizit. Fehlende Werte oder unerwartete
Kennungen führen zum Abbruch, nicht zu stiller Imputation.

In [ ]:
digest = verify_file(DATA_PATH)
data = pd.read_csv(DATA_PATH, usecols=COLUMNS).loc[:, COLUMNS]
validate_data(data)
assert isinstance(data, pd.DataFrame)
print(f"Form: {data.shape[0]} Zeilen × {data.shape[1]} Spalten")
print(f"SHA256: {digest}")
print("Fehlende Werte je Spalte:")
print(data.isna().sum().to_string())

## Die ersten zehn Zeilen

`head(10)` zeigt die ersten zehn Beobachtungen in Originalreihenfolge.
Kelvin, rpm und Nm bleiben in den Spaltennamen sichtbar.

In [ ]:
first_ten = data.head(10)
assert first_ten.shape == (10, 4)
display(first_ten)

## Grundstatistik

`describe()` zeigt Anzahl, Mittelwert, Standardabweichung, Minimum, Quartile und
Maximum. Die Statistik von UDI beschreibt nur die Nummerierung und ist keine
physikalische Kennzahl. Es werden keine Zeilen entfernt oder Werte verändert.

In [ ]:
summary = data.describe()
assert (summary.loc["count"] == 10000).all()
display(summary)
summary.to_csv(output_dir / "summary_statistics.csv")

## Drehmoment über UDI

Die Linie verbindet alle Beobachtungen in ihrer UDI-Reihenfolge. Sie visualisiert
die vom Auftrag geforderte Zuordnung, beweist aber keinen zeitlichen Verlauf.
Bei 10.000 Punkten ist Überlagerung zu erwarten; die Grafik dient einer ersten
Übersicht. Eine physikalische Interpretation würde zusätzliche Prozesskenntnis
und Zeitangaben benötigen.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
ax.plot(data["UDI"], data["Torque [Nm]"], linewidth=0.6, alpha=0.7, color="steelblue")
ax.set(title="AI4I: Drehmoment über UDI", xlabel="UDI (Zeilenkennung)", ylabel="Drehmoment [Nm]")
ax.grid(alpha=0.25)
fig.savefig(output_dir / "torque_over_udi.png", dpi=160)
display(fig)
plt.close(fig)

## Tatsächlich berechnete Zusammenfassung

Die folgenden Zahlen werden aus dem geladenen DataFrame berechnet, nicht von der
KI vorab eingesetzt. Es handelt sich um deskriptive Angaben zu synthetischen Daten.

In [ ]:
display(Markdown(
    f"Geprüft wurden **{len(data):,} Zeilen** und **{len(data.columns)} Spalten**. "
    f"In diesen Spalten fehlen **{int(data.isna().sum().sum())} Werte**. "
    f"Das mittlere Drehmoment beträgt **{data['Torque [Nm]'].mean():.3f} Nm** "
    f"(Minimum {data['Torque [Nm]'].min():.1f}, Maximum {data['Torque [Nm]'].max():.1f} Nm). "
    "Die Ergebnisse erlauben weder eine Kausalitätsaussage noch den Nachweis realer Maschinendynamik."
))

## Zelltypen und Export

- **Code:** wird vom Python-Kernel ausgeführt; Reihenfolge und Zustand beeinflussen Ergebnisse.
- **Markdown:** enthält Erläuterungen, Links und Formatierung; kein Python-Code.
- **Raw:** enthält unverarbeiteten Text, wird nicht vom Kernel ausgeführt; ob er in
  einem Export erscheint, hängt von Exportformat und Raw-MIME-Einstellung ab.

Die nächste Zelle ist tatsächlich vom Typ Raw. Der Notebook-Export wird durch
`uv run python scripts/build_exercise_10_2.py` reproduziert: frischer Kernel,
gespeicherte Ausgaben, HTML mit eingebetteter Grafik und PDF über Chromium.
Der alternative nbconvert-PDF-Weg über LaTeX ist nicht erforderlich.

## Fragen für die studentische Prüfung

1. Kannst du erklären, weshalb wir Originaleinheiten beibehalten und UDI nicht als Zeit interpretieren?
2. Stimmen Auswahl der vier Spalten, erste zehn Zeilen, Statistik und Achsen mit dem Auftrag überein?
3. Kannst du den Unterschied zwischen Code, Markdown und Raw im Notebook nachvollziehen?
4. Sind HTML/PDF einschließlich Tabellen und Grafik lesbar, und lässt sich die Ausführung reproduzieren?

**Keine studentische Freigabe dokumentiert. Keine automatische Abgabe.**